In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor
>>> case Tumor
	DEGs 20484
		Up (#12140)
		Dw (#8344)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    13
1                      IG_C_pseudogene     3
2                            IG_J_gene     9
3                      IG_J_pseudogene     1
4                            IG_V_gene   127
5                      IG_V_pseudogene    48
6                              Mt_tRNA    17
7                                  TEC   148
8                            TR_C_gene     6
9                            TR_D_gene     1
10                           TR_J_gene     8
11                           TR_V_gene    45
12                     TR_V_pseudogene     4
13                              lncRNA  4162
14                               miRNA   139
15                            misc_RNA    36
16              polymorphic_pseudogene     4
17        

In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'


verbose=True
cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)


-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"


### Calc expression

 - calc_file_expression_tumor_normal_gtex()
   - get_dic_expression_tumor_and_normal()
     - get_filtered_tables()
     - get_table_given_fileID()
   - prepare_normal_tumor_tables()

  
#### Tables in: root_disease / lfc

In [8]:
cbio.root_disease, cbio.root_lfc, cbio.filename_demo

(PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD'),
 PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc'),
 PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/clinical_and_demographics_for_paad_cptac_2021.tsv'))

### Get cases, subtypes and clin_demo tables

In [9]:
verbose=False
force=False

PROG_ID = 'TCGA'
psi_id = 'SKCM'
psi_id = 'BRCA'
psi_id = 'PAAD'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

df_cases, df_subt, df_clin_demo, df_case_bar = cbio.get_cases_and_subtypes(batch_size=200, force=force, verbose=verbose)

df_cases.shape, df_clin_demo.shape, df_case_bar.shape

((170, 27), (140, 20), (140, 2))

In [10]:
verbose=False
force=False

imax_tumor=200
imax_normal=100

df_tumor, df_normal, df_gtex_ctrl = cbio.calc_file_expression_tumor_normal_gtex(
            imax_tumor=imax_tumor, imax_normal=imax_normal, force=force, verbose=verbose)

print(df_tumor.shape[1], df_normal.shape[1], df_gtex_ctrl.shape[1])

There are 49 tumor and 20 normal Gene Expression tables
>> prepare_normal_tumor_tables()
>>> Processing normal data: 20
>>> Processing tumor data: 49
Error: could not find GTEx ID for CPTAC3 ID 'CPTAC3 - PAAD'
52 23 0


In [11]:
df_tumor.head(3)

,geneid,symbol,biotype,T-C3L-02890,T-C3L-03635,T-C3L-02701,T-C3L-04072,T-C3L-00589,T-C3L-03123,T-C3N-01383,...,T-C3L-03632,T-C3N-02589,T-C3N-03006,T-C3N-02996,T-C3N-03754,T-C3L-03639,T-C3L-02606,T-C3N-03665,T-C3N-03173,T-C3N-02696
0,ENSG00000000003,TSPAN6,protein_coding,2,2,0,0,1,1,1,...,0,0,0,0,1,0,0,0,2,1
1,ENSG00000000005,TNMD,protein_coding,12,10,14,1,12,2,5,...,9,6,10,11,14,31,5,1,6,6
2,ENSG00000000419,DPM1,protein_coding,9,9,4,9,13,5,7,...,8,8,10,10,8,2,1,0,9,16


In [12]:
df_normal.head(3)

,geneid,symbol,biotype,N-C3L-04072,N-C3L-00589,N-C3L-03123,N-C3L-04080,N-C3L-00640,N-C3N-01719,N-C3L-07033,...,N-C3N-01899,N-C3N-00517,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696
0,ENSG00000000003,TSPAN6,protein_coding,0,0,0,0,0,0,3,...,1,0,0,1,2,2,0,1,0,0
1,ENSG00000000005,TNMD,protein_coding,7,19,4,22,8,5,4,...,10,31,5,8,7,15,2,5,13,9
2,ENSG00000000419,DPM1,protein_coding,13,11,3,5,9,11,17,...,3,7,6,5,10,13,5,1,9,5


In [13]:
df_gtex_ctrl.head(3)

""


### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [14]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

1) prog_id CPTAC3, psi_id PAAD, primary_site Pancreas, disease_id pancreatic_ductal_adenocarcinoma - /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc There are 49 tumor and 20 normal Gene Expression tables
>> prepare_normal_tumor_tables()
>>> Processing normal data: 20
>>> Processing tumor data: 49
Error: could not find GTEx ID for CPTAC3 ID 'CPTAC3 - PAAD'
df_tumor (60616, 52)
>>> PAAD (60616, 52) (60616, 23)
2) prog_id TCGA, psi_id PAAD, primary_site Pancreas, disease_id pancreatic_adenocarcinoma - /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc There are 82 tumor and 2 normal Gene Expression tables
>> prepare_normal_tumor_tables()
>>> Processing normal data: 2
>>> Processing tumor data: 82
Error: could not find GTEx ID for TCGA ID 'TCGA - PAAD'
df_tumor (60616, 85)
>>> PAAD (60616, 85) (60616, 5)
There are 131 tumor samples and 22 control samples merging studies
Normal: Removing 1 columns with values > 427.9047825755396 -> ['N-TCGA-H6-8124']


>> dfn_tumor (60616, 134)
>> dfn_nor

In [21]:
dfn_tumor.iloc[:5, 75:]

,T-TCGA-2J-AABV,T-TCGA-HZ-8005,T-TCGA-FB-AAPS,T-TCGA-F2-A44G,T-TCGA-HZ-7920,T-TCGA-F2-A7TX,T-TCGA-2L-AAQL,T-TCGA-HV-A5A5,T-TCGA-IB-AAUW,T-TCGA-IB-7652,...,T-TCGA-HZ-7289,T-TCGA-3A-A9IN,T-TCGA-3A-A9IS,T-TCGA-2L-AAQM,T-TCGA-3A-A9IR,T-TCGA-3A-A9IV,T-TCGA-3A-A9IO,T-TCGA-2J-AABT,T-TCGA-H6-A45N,T-TCGA-3A-A9IJ
0,395,783,587,594,1474,1896,596,969,1262,1268,...,1221,210,18,155,36,84,182,323,666,192
1,0,0,25,0,11,11,2,0,7,1,...,1,87,4,2,0,1,8,1,1,0
2,290,1143,467,656,1083,704,605,543,823,1194,...,843,474,711,493,744,528,457,447,364,524
3,142,296,332,538,737,512,527,729,850,794,...,648,331,565,266,578,274,545,377,404,436
4,111,245,238,289,381,468,303,371,406,443,...,371,159,220,176,241,156,258,240,237,201


In [16]:
dfn_normal.head(3)

,geneid,symbol,biotype,N-C3L-04072,N-C3L-00589,N-C3L-03123,N-C3L-04080,N-C3L-00640,N-C3N-01719,N-C3L-07033,...,N-C3N-00517,N-C3N-03069,N-C3N-02765,N-C3L-07037,N-C3N-02589,N-C3N-02996,N-C3L-02606,N-C3N-03173,N-C3N-02696,N-TCGA-H6-A45N
0,ENSG00000000003,TSPAN6,protein_coding,0,0,0,0,0,0,3,...,0,0,1,2,2,0,1,0,0,168
1,ENSG00000000005,TNMD,protein_coding,7,19,4,22,8,5,4,...,31,5,8,7,15,2,5,13,9,3
2,ENSG00000000419,DPM1,protein_coding,13,11,3,5,9,11,17,...,7,6,5,10,13,5,1,9,5,521


### Batch effect correction and cpm normalization

In [ ]:
force=False
verbose=False

perc_min_samples=0.25; top_n=10_000

df_sel, df_cpm, df_gene_annot = cbio.calc_expression_and_batch(dfn_tumor=dfn_tumor, group='Tumor', 
                                                           perc_min_samples=perc_min_samples, top_n=top_n,
                                                           force=force, verbose=verbose)

dfn, df_gene_annot = cbio.calc_cpm_merge_turmor_and_normal(dfn_tumor=dfn_tumor, dfn_normal=dfn_normal, 
                                                                   perc_min_samples=perc_min_samples, top_n=top_n,
                                                                   verbose=verbose)

dfall = gdc.dfall
print(dfn.shape, dfall.shape)
dfall.tail(3)

In [ ]:
df_pca = cbio.calc_PCA(df_scaled, n_components=10, verbose=False)
df_pca.head(3)

In [ ]:
cbio.plot_PCA(df_pca)

In [ ]:
df_eval, df_samp_clusters = cbio.calc_best_cluster(df_pca, min_clusters=3, max_clusters=8)
df_eval

In [ ]:
df_samp_clusters.head(6)

### PCA-UMAP

In [ ]:
n_neighbors=3
min_dist=0.2

df_umap = cbio.calc_PCA_UMAP(df_pca, df_samp_clusters, n_neighbors=n_neighbors, min_dist=min_dist, metric="euclidean")
print(df_umap.cluster.unique())
df_umap.head(3)

In [ ]:
cbio.plot_PCA_UMAP(df_umap, n_neighbors=n_neighbors, min_dist=min_dist, figsize=(6, 5))

### Hierarchical clustering alternative

For 32 samples, this is often better than UMAP.

In [ ]:
cbio.plot_HCA_PCA(df_pca, figsize=(10, 8))

In [ ]:
# Cut tree into k clusters

df_cluster_hca = cbio.cut_HCA_PCA(df_pca=df_pca, n_clusters=3, method="ward", criterion="maxclust", verbose=True)
df_cluster_hca.head(6)

In [ ]:
cbio.plot_HCA_PCA_UMAP(df_umap, figsize=(10, 8))

In [ ]:
# Cut tree into k clusters

df_cluster_umap = cbio.cut_HCA_PCA_UMAP(df_umap=df_umap, n_clusters=3, method="ward", criterion="maxclust", verbose=True)
df_cluster_umap.head(6)

### Define cluster marker genes

This finds genes high in one cluster compared with all others.

In [ ]:
gene_annot = (
    dfg_filt[["geneid", "symbol"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

### cluster signatures

In [ ]:
df_cpm.head(2)

In [ ]:
dfc_log = np.log2(df_cpm + 1)
dfc_log.head(2)

In [ ]:
dfc_log = np.log2(df_cpm + 1)

dfall, dfsig = cbio.find_cluster_signature_genes(df_logcpm=dfc_log, df_samp_clusters=df_samp_clusters, gene_annot=gene_annot, lfc_cutoff=1.0, fdr_cutoff=0.05,)

In [ ]:
dfall.head(3)

In [ ]:
dfsig.cluster.unique()

In [ ]:
dfclu = cbio.write_clusters(dfall, dfsig, LFC_cutoff=3., FDR_cutoff=0.01, verbose=False)

dfclu

### CALC_DEGS -> build_counts_and_metadata()

In [ ]:
cdegs = CALC_DEGS(root_src=cbio.root_src, run_conda=False)
cbio.cdegs = cdegs

verbose=True
df_tumor, df_normal, msg = cbio.get_tumor_normal_tables(verbose=verbose)

In [ ]:
df_tumor.head(3)

In [ ]:
df_normal.head(3)

In [ ]:
df_counts, df_meta = cdegs.build_counts_and_metadata(
            df_tumor=df_tumor,
            df_normal=df_normal,
            how="inner"
        )

df_counts.head(3)

In [ ]:
df_meta

In [ ]:
dff, normal_samples, tumor_samples = build_df_exp_and_filter(df_counts=df_counts, df_meta=df_meta,
    gene_col = "geneid",
    equal_var = False,)   # False = Welch t-test, safer when n differs

print(len(df_counts), len(dff))

dff.head(3)

In [ ]:
import seaborn as sns

In [ ]:
cols = ['geneid'] + normal_samples + tumor_samples

dff2 = dff[cols].copy()
dff2.set_index('geneid', inplace=True)
dff2.head(3)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import zscore

figsize = (12,8)

title = 'Hierarchical Clustering of Expression Data'

# numeric matrix
dff2 = dff2.apply(pd.to_numeric, errors="coerce").fillna(0)
mat = np.log2(dff2 + 1)

# gene-wise z-score using pandas/numpy
row_mean = mat.mean(axis=1)
row_std = mat.std(axis=1)

mat_z = mat.sub(row_mean, axis=0).div(row_std.replace(0, np.nan), axis=0)
mat_z = mat_z.replace([np.inf, -np.inf], np.nan).fillna(0)

cg = sns.clustermap(
    mat_z,
    metric="correlation",
    method="average",
    figsize=figsize,
    cmap="viridis",
    cbar=True,
)

title = "Hierarchical Clustering of Expression Data"
cg.figure.suptitle(title, y=1.02)

plt.show()


In [ ]:
mat_z

### Development & tests

In [ ]:
def get_representative_geneids(
    dfs: list[pd.DataFrame],
    min_fraction: float = 0.75,
) -> pd.DataFrame:
    """
    Return genes present in more than min_fraction of dataframes.

    For 10 dataframes and min_fraction=0.75:
    strict >75% means present in at least 8 dataframes.

    Presence is counted once per dataframe, even if duplicated inside a dataframe.
    """

    gene_cols: list[str] = ["geneid", "symbol"]

    n = len(dfs)
    if n == 0:
        return pd.DataFrame(columns=gene_cols + ["n_dfs", "fraction"])

    min_count = math.floor(n * min_fraction) + 1  # strict > min_fraction

    counter = Counter()

    for df in dfs:
        missing = [c for c in gene_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns in dataframe: {missing}")

        genes_in_df = (
            df[gene_cols]
            .dropna(subset=gene_cols)
            .astype(str)
            .drop_duplicates()
        )

        counter.update(map(tuple, genes_in_df.to_numpy()))

    result = (
        pd.DataFrame(
            [(geneid, symbol, count) for (geneid, symbol), count in counter.items()],
            columns=gene_cols + ["n_dfs"],
        )
        .assign(fraction=lambda x: x["n_dfs"] / n)
        .query("n_dfs >= @min_count")
        .sort_values(["n_dfs"] + gene_cols, ascending=[False] + [True] * len(gene_cols))
        .reset_index(drop=True)
    )

    return result

In [ ]:
df_list = []

cols = ["geneid", "symbol", "biotype", "dfc"]


i = 0
for _, dfa in dic_tumor.items():
    if dfa is None or dfa.empty:
        continue

    i += 1
    # print(i, end=' ')
    if "gene_id" in dfa.columns:
        dfa = dfa.rename(columns={"gene_id": "geneid"})
    if "gene_type" in dfa.columns:
        dfa = dfa.rename(columns={"gene_type": "biotype"})

    dfa = dfa[cols]
    df_list.append(dfa)

In [ ]:
dfq = cbio.get_representative_geneids(df_list, min_fraction=0.75)

print(len(dfq))

dfq.head(3)

In [ ]:
lista = np.unique(dfq.geneid.to_list())
print(len(lista))
lista[:3]

In [ ]:
df_tumor = pd.DataFrame()

cols = ["geneid", "symbol", "dfc"]
common_cols = ["geneid", "symbol"]

imax_tumor=50

i = 0
for _, dfa in dic_tumor.items():
    if dfa is None or dfa.empty:
        continue

    i += 1
    # print(i, end=' ')
    if "gene_id" in dfa.columns:
        dfa = dfa.rename(columns={"gene_id": "geneid"})
    if "gene_type" in dfa.columns:
        dfa = dfa.rename(columns={"gene_type": "biotype"})

    dfa = dfa[cols]

    dfa = (
        dfa.dropna(subset=['geneid', 'symbol'])
        .drop_duplicates(['geneid', 'symbol'])
    )

    dfa = dfa.rename(columns={"dfc": f"tumor_{i}"})


    dfa = dfa[dfa.geneid.isin(lista)]
    dfa.reset_index(drop=True, inplace=True)

    if df_tumor.empty:
        df_tumor = dfa
    else:
        if i <= imax_tumor:
            df_tumor = df_tumor.merge(dfa, on=common_cols, how="outer")
            print(i, dfa.shape, df_tumor.shape)
        else:
            if verbose:
                print(">>> dfa", len(dfa), ",".join(dfa.symbol[:30]))
            break


In [ ]:
df_tumor.head(3)

In [ ]:
pdwritecsv(df_tumor, 'PPAD_tumor_expression.tsv', verbose=True)

### Cluster

In [ ]:
gene_cols = ["geneid", "symbol"]
sample_cols = [c for c in df_tumor.columns if c not in gene_cols]

dfc = (
    df_tumor[sample_cols]
    .apply(pd.to_numeric, errors="coerce")  # non-numeric -> NaN
    .fillna(0)                              # NaN -> 0
).copy()

dfg = df_tumor[gene_cols].copy()

dfc.index = df_tumor["geneid"]

# filter low-count genes
min_samples = int(0.25 * len(sample_cols))

print(f"sample_cols {len(sample_cols)} and min_samples")

keep = list ((dfc >= 10).sum(axis=1) >= min_samples)

dfc_filt = dfc.loc[keep]
dfg_filt = dfg.loc[keep]

print(dfc_filt.shape)

# normalize by library size

library_sizes = dfc_filt.sum(axis=0)

df_cpm = dfc_filt.div(library_sizes, axis=1) * 1_000_000

In [ ]:
df_cpm

In [ ]:
dfc_log = np.log2(df_cpm + 1)

# Select most variable genes

top_n = 5_000

gene_var = dfc_log.var(axis=1)

top_genes = (
    gene_var
    .sort_values(ascending=False)
    .head(top_n)
    .index
)

df_sel = dfc_log.loc[top_genes].T.copy()
print(df_sel.shape)
df_sel.head(3)


In [ ]:
# Scale genes
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df_scaled = StandardScaler().fit_transform(df_sel)
df_scaled


In [ ]:
pca = PCA(n_components=10, random_state=42)
df_pca = pca.fit_transform(df_scaled)

df_pca = pd.DataFrame(
    df_pca[:, :3],
    index=df_sel.index,
    columns=["PC1", "PC2", "PC3"]
)

print(pca.explained_variance_ratio_[:5])

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df_pca["PC1"], df_pca["PC2"], s=80)
for sample in df_pca.index:
    plt.text(df_pca.loc[sample, "PC1"], df_pca.loc[sample, "PC2"], sample, fontsize=8)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of tumor samples")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_results = []

for k in range(3, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = model.fit_predict(df_pca)

    sil = silhouette_score(df_pca, labels)

    cluster_results.append({
        "k": k,
        "silhouette": sil,
        "labels": labels
    })

df_eval = pd.DataFrame([
    {"k": r["k"], "silhouette": r["silhouette"]}
    for r in cluster_results
])

df_eval

In [ ]:
best = max(cluster_results, key=lambda x: x["silhouette"])

df_samp_clusters = pd.DataFrame({
    "sample": df_sel.index,
    "cluster": best["labels"] + 1
})

df_samp_clusters

In [ ]:
# PCA-UMAP

import umap

reducer = umap.UMAP(
    n_neighbors=5,
    min_dist=0.2,
    metric="euclidean",
    random_state=42
)

X_umap = reducer.fit_transform(df_pca)

df_umap = pd.DataFrame(
    X_umap,
    index=df_sel.index,
    columns=["UMAP1", "UMAP2"]
)

df_umap = df_umap.merge(
    df_samp_clusters,
    left_index=True,
    right_on="sample",
    how="left"
)

df_umap

### Hierarchical cluster

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

Z = linkage(df_pca, method="ward")

plt.figure(figsize=(10, 5))
dendrogram(Z, labels=df_sel.index.tolist(), leaf_rotation=90)
plt.title("Hierarchical clustering of tumor samples")
plt.tight_layout()
plt.show()

In [ ]:
df_pca.head(2)

In [ ]:
df_umap.head(2)

In [ ]:
df2 = df_umap[ ['sample', 'UMAP1', 'UMAP2'] ]
df2.set_index('sample', inplace=True)

Z = linkage(df2, method="ward")

plt.figure(figsize=(10, 5))
dendrogram(Z, labels=df2.index.tolist(), leaf_rotation=90)
plt.title("PCA-UMAP Hierarchical clustering of tumor samples")
plt.tight_layout()
plt.show()

### Cut into k clusters

In [ ]:
k = 5

hc_labels = fcluster(Z, t=k, criterion="maxclust")

df_samp_clust_hc = pd.DataFrame({
    "sample": df_sel.index,
    "cluster": hc_labels
})

print( df_samp_clust_hc.groupby("cluster").size() )

df_samp_clust_hc

### cluster signatures

In [ ]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

def find_cluster_signature_genes(
    df_logcpm: pd.DataFrame,
    df_samp_clusters: pd.DataFrame,
    gene_annot: pd.DataFrame,
    sample_col: str = "sample",
    cluster_col: str = "cluster",
    min_logfc: float = 1.0,
    max_fdr=0.05,
):
    """
    Find marker/signature genes for each cluster.

    df_logcpm:
        genes x samples matrix, log2(CPM + 1)

    df_samp_clusters:
        dataframe with columns: sample, cluster

    gene_annot:
        optional dataframe with geneid, symbol
    """

    results = []

    for cluster_id in sorted(df_samp_clusters[cluster_col].unique()):

        in_samples  = df_samp_clusters.loc[df_samp_clusters[cluster_col] == cluster_id, sample_col].tolist()
        out_samples = df_samp_clusters.loc[df_samp_clusters[cluster_col] != cluster_id, sample_col].tolist()

        # keep only samples present in expression matrix
        in_samples  = [s for s in in_samples  if s in df_logcpm.columns]
        out_samples = [s for s in out_samples if s in df_logcpm.columns]

        if len(in_samples) < 2 or len(out_samples) < 2:
            print(f"Skipping cluster {cluster_id}: too few samples")
            continue

        df_mean_in = df_logcpm[in_samples].mean(axis=1)
        df_mean_out = df_logcpm[out_samples].mean(axis=1)

        df_lfc = df_mean_in - df_mean_out

        pvals = []

        for geneid in df_logcpm.index:
            stat, p = ttest_ind(
                df_logcpm.loc[geneid, in_samples],
                df_logcpm.loc[geneid, out_samples],
                equal_var=False,
                nan_policy="omit",
            )
            pvals.append(p)

        fdr = multipletests(pvals, method="fdr_bh")[1]

        res = pd.DataFrame({
            "geneid": df_logcpm.index,
            "cluster": cluster_id,
            "n_in": len(in_samples),
            "n_out": len(out_samples),
            "mean_in": df_mean_in.values,
            "mean_out": df_mean_out.values,
            "lfc": df_lfc.values,
            "pvalue": pvals,
            "fdr": fdr,
        })

        if gene_annot is not None:
            res = res.merge(gene_annot, on="geneid", how="left")

        res = res.sort_values(
            ["lfc", "fdr"],
            ascending=[False, True]
        )

        results.append(res)

    all_results = pd.concat(results, ignore_index=True)

    signatures = (
        all_results
        .query("lfc >= @min_logfc and fdr <= @max_fdr")
        .sort_values(["cluster", "lfc", "fdr"], ascending=[True, False, True])
        .reset_index(drop=True)
    )

    return all_results, signatures

In [ ]:
lista = np.unique(dfall.cluster)

LFC_cutoff=3
FDR_cutoff=1e-3

for ncluster in lista:
    df2 = dfsig[dfsig.cluster == ncluster]
    df2 = df2[ (df2['lfc'].abs() > LFC_cutoff) & (df2['fdr'] < FDR_cutoff) ]
    print(len(df2))

    write_txt('\n'.join(df2.symbol), f"cluster_{ncluster}_signature_genes.txt")

df2

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

def build_df_exp_and_filter(
    df_counts: pd.DataFrame,
    df_meta: pd.DataFrame,
    gene_col: str = "geneid",
    condition_col: str = "condition",
    sample_col: str = "sample",
    tumor_label: str = "tumor",
    normal_label: str = "normal",
    equal_var: bool = False,   # False = Welch t-test, safer when n differs
) -> tuple[pd.DataFrame, list, list]:
    df = df_counts.copy()

    # Samples by condition
    normal_samples = df_meta.loc[
        df_meta[condition_col] == normal_label, sample_col
    ].tolist()

    tumor_samples = df_meta.loc[
        df_meta[condition_col] == tumor_label, sample_col
    ].tolist()

    # Keep only samples present in df_counts
    normal_samples = [s for s in normal_samples if s in df.columns]
    tumor_samples = [s for s in tumor_samples if s in df.columns]

    sample_cols = normal_samples + tumor_samples

    ncols_normal = len(normal_samples)
    ncols_tumor  = len(tumor_samples)

    nmin_cols = min(ncols_normal, ncols_tumor)

    df["total"] = df[sample_cols].sum(axis=1)

    df = df.loc[
        df["total"] > nmin_cols * 25
    ].reset_index(drop=True, inplace=False)



    # Convert counts to numeric
    df[normal_samples + tumor_samples] = df[normal_samples + tumor_samples].apply(
        pd.to_numeric, errors="coerce"
    )

    # Optional but recommended for RNA-seq counts:
    # log-transform before t-test
    normal_mat = np.log2(df[normal_samples] + 1)
    tumor_mat = np.log2(df[tumor_samples] + 1)

    # Row-wise t-test: tumor vs normal
    t_stat, pval = ttest_ind(
        tumor_mat,
        normal_mat,
        axis=1,
        equal_var=equal_var,
        nan_policy="omit",
    )

    df["t_stat"] = t_stat
    df["pval"] = pval

    # Useful summaries
    df["mean_normal"] = normal_mat.mean(axis=1)
    df["mean_tumor"] = tumor_mat.mean(axis=1)
    df["lfc"] = df["mean_tumor"] - df["mean_normal"]
    df["abs_lfc"] = df["lfc"].abs()

    # Order by p-value
    df = df.sort_values("pval", ascending=True)

    # Keep the 40% lowest p-values
    df = df[df.lfc < 0.01]
    df.reset_index(drop=True, inplace=True)

    return df, normal_samples, tumor_samples
